In [ ]:
from typing import Iterator
import numpy as np
import pyquist as pq


def iter_frames(audio: pq.Audio, hop_length: int, frame_length: int) -> Iterator[np.ndarray]:
    for start in range(0, len(audio) - frame_length + 1, hop_length):
        yield audio.samples[start:start + frame_length]


def overlap_add(frames: np.ndarray, hop_length: int, sample_rate: int) -> pq.Audio:
    num_frames, frame_length, num_channels = frames.shape
    out = np.zeros((hop_length * (num_frames - 1) + frame_length, num_channels), dtype=frames.dtype)
    for k, frame in enumerate(frames):
        out[k * hop_length:k * hop_length + frame_length] += frame
    return pq.Audio(out, sample_rate)

from pyquist.helper import frequency_to_pitch

In [ ]:
# A tiny monophonic transcriber: for each frame, find the loudest frequency,
# round it to the nearest musical pitch, and emit a note when the pitch changes.
audio = pq.Audio.from_file("../assets/audio-melody.wav")
sr = audio.sample_rate
N_F, N_H = 4096, 1024

# One magnitude spectrum per frame: window each frame, then take its DFT.
frames = np.array(list(iter_frames(audio, N_H, N_F)))[:, :, 0]   # (num_frames, N_F), mono
S = np.abs(np.fft.rfft(frames * np.hanning(N_F), axis=1))
freqs = np.fft.rfftfreq(N_F, 1 / sr)
seconds_per_frame = N_H / sr
threshold = 0.05 * S.max()

events, current, onset = [], None, 0.0
for k in range(len(S)):
    if S[k].max() < threshold:                   # silence
        pitch = None
    else:
        pitch = int(round(frequency_to_pitch(freqs[np.argmax(S[k])])))
    if pitch != current:                         # the note changed
        if current is not None:
            events.append((onset, {"pitch": current, "duration": k * seconds_per_frame - onset}))
        current, onset = pitch, k * seconds_per_frame
if current is not None:                          # flush the final note
    events.append((onset, {"pitch": current, "duration": len(S) * seconds_per_frame - onset}))

score = pq.Score(events)
for event in score:
    print(f"t = {event.time:4.2f}s   MIDI pitch {event.kwargs['pitch']}")